# 05 — Column Mapping Pattern

Purpose: map source schema to target schema using config instead of hardcoding select/alias logic.

Process schema:

SOURCE DATA
  |
  v
COLUMN MAPPING CONFIG
  |
  v
SELECT + ALIAS
  |
  v
TARGET DATA

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("column-mapping-pattern")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Source data

In [2]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("user", StringType(), True),
    StructField("amt", DoubleType(), True),
])

rows = [
    ("e1","u1",10.0),
    ("e2","u2",20.0),
]

df = spark.createDataFrame(rows, schema)
df.show()

+---+----+----+
| id|user| amt|
+---+----+----+
| e1|  u1|10.0|
| e2|  u2|20.0|
+---+----+----+



## Step 2 — Mapping config

In [3]:
mapping = {
    "id": "event_id",
    "user": "user_id",
    "amt": "amount"
}

mapping

{'id': 'event_id', 'user': 'user_id', 'amt': 'amount'}

## Step 3 — Apply mapping

In [4]:
mapped_df = df.select([
    F.col(src).alias(dst) for src, dst in mapping.items()
])

mapped_df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|     u2|  20.0|
+--------+-------+------+



## Step 4 — Partial mapping (subset)

In [5]:
partial_mapping = {
    "id": "event_id",
    "amt": "amount"
}

partial_df = df.select([
    F.col(src).alias(dst) for src, dst in partial_mapping.items()
])

partial_df.show()

+--------+------+
|event_id|amount|
+--------+------+
|      e1|  10.0|
|      e2|  20.0|
+--------+------+



## Step 5 — Missing column handling

In [6]:
safe_mapping = {
    "id": "event_id",
    "user": "user_id",
    "missing_col": "x"
}

safe_df = df.select([
    F.col(src).alias(dst) if src in df.columns else F.lit(None).alias(dst)
    for src, dst in safe_mapping.items()
])

safe_df.show()

+--------+-------+----+
|event_id|user_id|   x|
+--------+-------+----+
|      e1|     u1|NULL|
|      e2|     u2|NULL|
+--------+-------+----+



## Step 6 — Control totals

In [7]:
totals = spark.createDataFrame(
    [
        ("input_rows", df.count()),
        ("mapped_rows", mapped_df.count())
    ],
    ["metric","value"]
)

totals.show()

+-----------+-----+
|     metric|value|
+-----------+-----+
| input_rows|    2|
|mapped_rows|    2|
+-----------+-----+



In [8]:
spark.stop()